In [1]:
from typing import Any

from pytorch_lightning.utilities.types import STEP_OUTPUT

""" Class 25 | Project 2 | Machine Translation using Pretrained Model

Objectives:
1. End-to-end machine translation training pipeline
2. Fine-tune a pre-trained model for the custom dataset
"""

import pytorch_lightning as pl
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from torchmetrics.text import BLEUScore
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

ModuleNotFoundError: No module named 'pytorch_lightning'

In [ ]:
# MLFlow Integration
import mlflow
import mlflow.pytorch
from pytorch_lightning.loggers import MLFlowLogger
from datetime import datetime

print(f"MLFlow version: {mlflow.__version__}")

# Set tracking URI (local ./mlruns by default)
# For remote tracking, set: mlflow.set_tracking_uri("http://<host>:5000")
mlflow.set_tracking_uri("file:./mlruns")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

NameError: name 'torch' is not defined

In [ ]:
"""Task: English to Bangla """

mt_pretrained_model_name = "shhossain/opus-mt-en-to-bn"

In [ ]:
""" For NLP tasks, we basically need two entities:
1. Tokenizer
2. Model
"""

tokenizer = AutoTokenizer.from_pretrained(mt_pretrained_model_name)
mt_pretrained_model = AutoModelForSeq2SeqLM.from_pretrained(mt_pretrained_model_name)

NameError: name 'AutoTokenizer' is not defined

# Data

In [ ]:
"""
Sentence: How are you, dude?
Tokens: 'How', 'are', 'you', 'dude?'
ids: 125, 14, 145, 78
max_length = 3
ids: [125, 14, 145]
"""

class MTDataset(Dataset):
    def __init__(self, csv_file):
        self.data = pd.read_csv(csv_file)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        src_text = str(self.data.iloc[idx]['en'])
        tgt_text = str(self.data.iloc[idx]['bn'])

        src_encoding = tokenizer(
            src_text,
            max_length=128,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )

        tgt_encoding = tokenizer(
            tgt_text,
            max_length=128,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'src_input_ids': src_encoding['input_ids'].squeeze(),
            'src_attention_mask': src_encoding['attention_mask'].squeeze(),
            'tgt_input_ids': tgt_encoding['input_ids'].squeeze(),
            'tgt_attention_mask': tgt_encoding['attention_mask'].squeeze()
        }

"""
example: How are you, dude?
input_ids: 125, 14, 145, 78
max_length = 7
input_ids: [125, 14, 145, 147, 0, 0, 0]
attention_mask: [1, 1, 1, 1, 0, 0, 0]
"""

NameError: name 'Dataset' is not defined

In [ ]:
class MTDataModule(pl.LightningDataModule):
    def __init__(self, train_csv, val_csv, test_csv, batch_size=32):
        super().__init__()
        self.train_csv = train_csv
        self.val_csv = val_csv
        self.test_csv = test_csv
        self.batch_size = batch_size

    def setup(self, stage=None):
        self.train_dataset = MTDataset(self.train_csv)
        self.val_dataset = MTDataset(self.val_csv)
        self.test_dataset = MTDataset(self.test_csv)

    def train_dataloader(self):
        return DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            shuffle=True
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_dataset,
            batch_size=self.batch_size,
            shuffle=False
        )

    def test_dataloader(self):
        return DataLoader(
            self.test_dataset,
            batch_size=self.batch_size,
            shuffle=False
        )

In [ ]:
data_module = MTDataModule(
    train_csv=r'/content/train.csv',
    val_csv=r'C:\Users\ASUS\Downloads\val.csv',
    test_csv=r'C:\Users\ASUS\Downloads\test.csv',
    batch_size=32
)

# Model

In [ ]:
class MTModel(pl.LightningModule):
    def __init__(self):
        super().__init__()
        # load pretrained model
        self.model = AutoModelForSeq2SeqLM.from_pretrained(mt_pretrained_model_name)
        # load pretrained tokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(mt_pretrained_model_name)
        # learning rate
        self.learning_rate = 2e-5
        # loss function
        self.loss_fn = nn.CrossEntropyLoss(
            ignore_index=self.tokenizer.pad_token_id
        )
        # evaluation metric
        self.bleu = BLEUScore()

    def forward(self,
                src_input_ids,
                src_attention_mask,
                tgt_input_ids,
                tgt_attention_mask
        ):
        outputs = self.model(
            input_ids=src_input_ids,
            attention_mask=src_attention_mask,
            decoder_input_ids=tgt_input_ids[:, :-1],
            decoder_attention_mask=tgt_attention_mask[:, :-1]
        )
        return outputs

    def training_step(self, batch, batch_idx):
        loss = self.compute_loss(batch, batch_idx, 'train')
        self.log('train_loss', loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        loss = self.compute_loss(batch, batch_idx, 'val')
        self.log('val_loss', loss, prog_bar=True)
        return loss

    def test_step(self, batch, batch_idx):
        loss = self.compute_loss(batch, batch_idx, 'test')
        self.log('test_loss', loss, prog_bar=True)
        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.learning_rate)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=10
        )
        return {'optimizer': optimizer, 'lr_scheduler': scheduler}

    def compute_loss(self, batch, batch_idx, stage):
        src_input_ids = batch['src_input_ids']
        src_attention_mask = batch['src_attention_mask']
        tgt_input_ids = batch['tgt_input_ids']
        tgt_attention_mask = batch['tgt_attention_mask']

        outputs = self(
            src_input_ids,
            src_attention_mask,
            tgt_input_ids,
            tgt_attention_mask
        )
        logits = outputs.logits
        loss = self.loss_fn(
            logits.view(-1, logits.size(-1)),
            tgt_input_ids[:, 1:].contiguous().view(-1)
        )

        if stage == 'val' or stage == 'test':
            preds = torch.argmax(logits, dim=-1)
            pred_texts = self.tokenizer.batch_decode(preds, skip_special_tokens=True)
            tgt_texts = self.tokenizer.batch_decode(tgt_input_ids[:, 1:], skip_special_tokens=True)
            bleu_score = self.bleu(pred_texts, [[tgt] for tgt in tgt_texts])
            self.log(f'{stage}_bleu', bleu_score, prog_bar=True)

        return loss


NameError: name 'pl' is not defined

In [ ]:
model = MTModel()

# Train

In [ ]:
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping, LearningRateMonitor

# Callbacks
checkpoint_callback = ModelCheckpoint(
    monitor="val_bleu",
    mode="max",
    save_top_k=2,
    filename="mt-en-bn-{epoch:02d}-{val_bleu:.3f}",
    save_weights_only=False,
)
early_stop_callback = EarlyStopping(monitor="val_loss", patience=3, mode="min")
lr_monitor = LearningRateMonitor(logging_interval="epoch")

trainer = pl.Trainer(
    max_epochs=2,
    accelerator='gpu' if torch.cuda.is_available() else 'cpu',
    devices=1,
    precision=16,
    log_every_n_steps=10,
    val_check_interval=0.25,
    logger=mlflow_logger,  # <-- MLFlow logger
    callbacks=[checkpoint_callback, early_stop_callback, lr_monitor],
)

In [ ]:
# MLFlow Experiment Setup
experiment_name = "mt_en_bn_finetune"
run_name = f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

mlflow_logger = MLFlowLogger(
    experiment_name=experiment_name,
    run_name=run_name,
    tracking_uri="file:./mlruns",
    log_model=True,  # auto-log the LightningModule
)

# Log hyperparameters
mlflow_logger.log_hyperparams({
    "model_name": mt_pretrained_model_name,
    "max_epochs": 2,
    "batch_size": 32,
    "learning_rate": 2e-5,
    "max_length": 128,
    "precision": 16,
})

print(f"MLFlow Experiment: '{experiment_name}'")
print(f"Run Name: {run_name}")
print(f"Run ID: {mlflow_logger.run_id}")

In [ ]:
trainer.fit(model, data_module)

In [ ]:
trainer.test(model, data_module)

In [ ]:
# Save and Log Model Artifacts to MLFlow
import os

save_path = "./saved_model"
os.makedirs(save_path, exist_ok=True)

# Save model and tokenizer to disk
model.model.save_pretrained(save_path)
model.tokenizer.save_pretrained(save_path)
print(f"Model saved to: {save_path}")

# Log model artifacts to MLFlow
mlflow_logger.experiment.log_artifact(
    run_id=mlflow_logger.run_id,
    local_path=save_path,
    artifact_path="final_model"
)
print("Model artifacts logged to MLFlow.")

# End the MLFlow run
mlflow.end_run()
print(f"MLFlow run '{run_name}' ended.")
print(f"View results: mlflow ui --port 5000")

In [ ]:
# Demo: Translate a sentence with the fine-tuned model

def translate_text(text, model, tokenizer, max_length=128):
    """Translate a single English sentence to Bangla."""
    model.eval()
    encoding = tokenizer(
        text,
        max_length=max_length,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    )
    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)
    
    with torch.no_grad():
        generated_ids = model.model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=max_length,
            num_beams=4,
            early_stopping=True,
        )
    return tokenizer.decode(generated_ids[0], skip_special_tokens=True)

# Test translations
test_sentences = [
    "The man is walking in the park.",
    "A dog is playing with a ball.",
    "She is reading a book.",
]

for sentence in test_sentences:
    translation = translate_text(sentence, model, tokenizer)
    print(f"EN: {sentence}")
    print(f"BN: {translation}")
    print("-" * 40)

In [ ]:
model.model.config

NameError: name 'model' is not defined

In [ ]:
for name, module in model.model.named_modules():
    print(name)

NameError: name 'model' is not defined